In [11]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [13]:
N_SIMS = 200000  # Número de simulaciones de Monte Carlo
N_YEARS = 10  # Horizonte del DCF en años (n años hacia el futuro)
DT_SIM = 1/12  # Paso de tiempo de simulación en años (1.0 = anual, 1/12 = mensual)
WACC = 0.10  # Tasa de descuento r = 10% (0.10)

# Volúmenes anuales de producción/venta (ajusta según el tamaño de tu empresa)
OIL_PROD_ANNUAL = 1_000_000  # Barriles de petróleo vendidos por año (bbl/año)
GAS_PROD_ANNUAL = 5_000_000  # Millones de BTU de gas vendidos por año (mmBTU/año)

Para estimar la volatilidad ($\sigma$) y la correlación ($\rho$) con precisión estadística: Si calibraras usando solo 1 o 2 años de datos mensuales (apenas 12 o 24 puntos), tu muestra sería demasiado pequeña y quedaría sesgada por lo que haya pasado en ese año puntual (por ejemplo, si fue un año anormalmente tranquilo, subestimarías el riesgo para el próximo año). Con 20 años (~240 observaciones mensuales), tienes suficiente tamaño muestral para que la desviación estándar de los residuos y la correlación entre gas y petróleo sean estadísticamente robustas.

In [18]:
HIST_START = '2005-01-01' 
HIST_END = '2026-11-01'

En los mercados financieros globales, el petróleo y el gas natural no cotizan como acciones en la bolsa (como Apple o Ecopetrol); la referencia de precio mundial que aparece en las noticias y en Yahoo Finance es siempre el contrato de futuros a 1 mes (front-month). **En la práctica financiera, el futuro a 1 mes (=F) es el precio de mercado actual (spot).**

Aquí =F es simplemente el sufijo genérico de Yahoo Finance que significa Futures (contrato continuo del mes más cercano, sin importar en qué mes del año estemos hoy). A diferencia de una acción que compras en un clic, el petróleo crudo y el gas natural son bienes físicos pesados que requieren oleoductos, buques y almacenamiento. Por eso no existe un "precio spot instantáneo" único en Yahoo Finance; el precio que todo el mundo toma como referencia diaria del petróleo WTI o del gas Henry Hub es el contrato de entrega más próximo (CL=F y NG=F)

In [29]:
tickers = ['CL=F', 'NG=F']  # CL=F: Petróleo WTI | NG=F: Gas Natural Henry Hub
raw_data = yf.download(tickers, 
                       start=HIST_START, 
                       end=HIST_END, 
                       interval='1mo', 
                       progress=False)['Close']

df_prices = raw_data.rename(columns={'CL=F': 'Oil', 'NG=F': 'Gas'}).dropna()

`tickers = ['CL=F', 'NG=F']`: Define una lista con los dos símbolos bursátiles que usará Yahoo Finance:

`CL=F`: Es el futuro continuo (front-month, equivalente al concepto C1 que vimos antes) del Petróleo Crudo WTI (West Texas Intermediate, referencia en EE. UU., cotizado en dólares por barril).

`NG=F`: Es el futuro continuo del Gas Natural Henry Hub (cotizado en dólares por mmBTU).

`interval='1mo'`: Pide los datos agrupados con frecuencia mensual (un dato por mes), tal como se recomienda para calibrar modelos de valoración DCF a largo plazo y evitar el ruido diario.

TODO: averiguar cómo acceder el precio spot desde FRED.

In [30]:
df_prices

Ticker,Oil,Gas
Date,,
2005-01-01,48.200001,6.321
2005-02-01,51.750000,6.730
2005-03-01,55.400002,7.653
2005-04-01,49.720001,6.585
2005-06-01,56.500000,6.981
...,...,...
2026-06-01,69.500000,3.275
2026-07-01,84.669998,2.747
2026-08-01,85.760002,2.935


In [31]:
fig_eda = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        'Histórico Mensual: Petróleo WTI ($/bbl)',
        'Histórico Mensual: Gas Natural Henry Hub ($/mmBTU)',
    ),
)

# Serie histórica del Petróleo
fig_eda.add_trace(
    go.Scatter(
        x=df_prices.index,
        y=df_prices['Oil'],
        mode='lines',
        line=dict(color='#00bfff', width=2),
        name='Petróleo WTI ($/bbl)',
    ),
    row=1,
    col=1,
)

# Serie histórica del Gas Natural
fig_eda.add_trace(
    go.Scatter(
        x=df_prices.index,
        y=df_prices['Gas'],
        mode='lines',
        line=dict(color='#00e676', width=2),
        name='Gas Natural ($/mmBTU)',
    ),
    row=2,
    col=1,
)

fig_eda.update_yaxes(title_text='Precio ($/bbl)', row=1, col=1)
fig_eda.update_yaxes(title_text='Precio ($/mmBTU)', row=2, col=1)
fig_eda.update_xaxes(title_text='Fecha', row=2, col=1)

fig_eda.update_layout(
    title='Análisis Exploratorio (EDA): Evolución Histórica de Precios (2005–2026)',
    template='plotly_dark',
    width=950,
    height=650,
    hovermode='x unified',
    showlegend=False,
)

fig_eda.show()

Comparar las curvas en BTU.

Para que la regresión AR(1) detecte la reversión a la media ($\theta$ y $\mu$): La reversión a la media en materias primas es un fenómeno lento (los ciclos del petróleo y el gas suelen tardar entre 2 y 5 años en subir y volver a su equilibrio). En una ventana corta de 1 o 2 años, el precio muchas veces parece una línea recta hacia arriba o hacia abajo (tendencia), lo que hace que la pendiente $b$ de la regresión sea $\ge 1$ y el modelo falle al intentar calcular $\theta = -\ln(b)/\Delta t$.

Porque Cuando $\theta$ es positivo (ej. $\theta = 0.5$): El exponente $-\theta \Delta t$ es negativo, por lo que $e^{-\theta \Delta t}$ es un número menor que $1$ (ej. $0.96$). Eso actúa como un freno que acerca $X_t$ hacia la media $\mu$.Cuando $\theta$ es negativo (ej. $\theta = -0.5$): El signo menos de la fórmula cancela el menos de $\theta$, volviendo el exponente positivo ($e^{+0.5 \Delta t} > 1$). En lugar de atraer el precio hacia el equilibrio $\mu$. Ejemplo en geogebra: https://www.geogebra.org/classic/ddpsykxd.

In [44]:
# Ejemplo de calibración de un proceso Ornstein-Uhlenbeck (OU) mediante regresión lineal OLS
X = [4.1, 4.3, 4.2, 4.5]
print(f"Datos de X_t: {X[1:]}")
print(f"Datos de X_{{t-1}}: {X[:-1]}")
X_t = X[1:]
A = sm.add_constant(X[:-1])
print(f"Tiempos actuales:\n {X_t}")
print(f"Matriz A:\n {A}")
model = sm.OLS(X_t, A).fit()
b, m = model.params
resid = model.resid

sd_eps = np.std(resid, ddof=2)
print(f"Parámetros calibrados: b = {b:.4f}, m = {m:.4f}")
print(f"Residuos de la regresión: {resid}")
print(f"Desviación estándar de los residuos: sd_eps = {sd_eps:.4f}")

Datos de X_t: [4.3, 4.2, 4.5]
Datos de X_{t-1}: [4.1, 4.3, 4.2]
Tiempos actuales:
 [4.3, 4.2, 4.5]
Matriz A:
 [[1.  4.1]
 [1.  4.3]
 [1.  4.2]]
Parámetros calibrados: b = 6.4333, m = -0.5000
Residuos de la regresión: [-0.08333333 -0.08333333  0.16666667]
Desviación estándar de los residuos: sd_eps = 0.2041


\begin{align*}
X_2 &= b + mX_1 \\
X_3 &= b + mX_2 \\
\vdots \;&\;\;\;\vdots \qquad\;\;\vdots \\
X_n &= b + mX_{n-1} 
\end{align*}

donde 
\begin{align*}
X_1 &: \text{2005-01} \\
X_2 &: \text{2005-02} \\
&\;\;\vdots \\
X_{n} &: \text{2026-11}
\end{align*}



\begin{equation}
\mathbf{X_t} = b\mathbf{1}+m\mathbf{X_{t-1}} \tag{6.2.2}
\end{equation}

donde
\begin{equation}
\mathbf{y} = \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{pmatrix}, \;
A = \begin{pmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_n \end{pmatrix} \text{ y } \;
\mathbf{u} = \begin{pmatrix} b \\ m \end{pmatrix} \tag{6.2.3}
\end{equation}

\begin{align*}
X_2 &= b + mX_1 + \epsilon(2) \quad \text{variable aleatoria - epsilon normal(0,sigma)} \\
\hat{X}_2 &= b + mX_1 \quad \text{estimador} \\ 
X_2 &= \hat{X}_2 + \epsilon(2) \\
X_2 &- \hat{X}_2 = \epsilon(2)
\end{align*}

In [ ]:
def calibrate_ou_ols(price_series, dt_hist=1 / 12):
    """
    Calibra un proceso Ornstein-Uhlenbeck (OU) sobre el logaritmo natural del precio:
    dX_t = theta * (mu - X_t) dt + sigma * dW_t, 
    donde X_t = ln(P_t)
    """

    # X = np.log(price_series.values): Convierte los precios en dólares (P_t) a logaritmos naturales 
    # X_t = ln P_t = ln (P_t/1 USD). Así se garantiza que al devolverlos a dólares más adelante con e^{X_t}, 
    # ningún precio simulado pueda ser negativo.
    X = np.log(price_series.values)
    X_t = X[1:]
    X_prev = sm.add_constant(X[:-1])
    model = sm.OLS(X_t, X_prev).fit() # X_t = a*m(1) + b*X_{t-1} + eps_t 
    
    # a (el intercepto): Es el valor constante que multiplica a la columna de unos (1.0).
    # b (la pendiente): Es el coeficiente que multiplica a $X_{t-1}$ 
    a, b = model.params

    # resid es la diferencia exacta en cada mes entre el precio real en la historia X_t 
    # y el precio que la recta predijo \hat{X}_t = a + b X_{t-1}$ 
    resid = model.resid

    # Calculamos la desviación estándar de esos residuos mensuales, que mide qué tan grandes son las sorpresas típicas 
    # del mercado cada mes.
    sd_eps = np.std(resid, ddof=2)

    # Validar reversión a la media (0 < m < 1)
    if not (0 < m < 1):
        print(
            f'Advertencia: m={m:.4f} fuera de (0,1). La serie muestra tendencia o raíz unitaria.'
        )

    # Conversión exacta de AR(1) discreto a parámetros continuos OU
    theta = -np.log(m) / dt_hist
    mu = b / (1 - m)
    sigma = sd_eps * np.sqrt((-2 * np.log(m)) / (dt_hist * (1 - m**2)))

    # Precio de equilibrio de largo plazo en dólares (con ajuste de Jensen)
    eq_price = np.exp(mu + (sigma**2) / (4 * theta))

    return {
        'theta': theta,
        'mu': mu,
        'sigma': sigma,
        'eq_price': eq_price,
        'last_price': price_series.iloc[-1],
        'resid': resid,
    }

\begin{align*}
\theta &= -\frac{\ln(m)}{\Delta t_{\text{hist}}} \\
\mu &= \frac{b}{1 - m} \\
\sigma &= \sigma_{\varepsilon} \sqrt{\frac{-2 \ln(m)}{\Delta t_{\text{hist}} (1 - m^2)}}
\end{align*}

In [9]:
oil_params = calibrate_ou_ols(df_prices['Oil'], dt_hist=1 / 12)
gas_params = calibrate_ou_ols(df_prices['Gas'], dt_hist=1 / 12)

# Calcular correlación histórica entre los choques (residuos) de Petróleo y Gas
rho = np.corrcoef(oil_params['resid'], gas_params['resid'])[0, 1]

print('\n--- PARÁMETROS CALIBRADOS (PROCESO OU LOGARÍTMICO) ---')
print(
    f"Petróleo (WTI) -> Actual: ${oil_params['last_price']:.2f} | Equilibrio LP: ${oil_params['eq_price']:.2f} | theta: {oil_params['theta']:.4f} | sigma: {oil_params['sigma']:.2%}"
)
print(
    f"Gas Natural    -> Actual: ${gas_params['last_price']:.2f} | Equilibrio LP: ${gas_params['eq_price']:.2f} | theta: {gas_params['theta']:.4f} | sigma: {gas_params['sigma']:.2%}"
)
print(f'Correlación entre choques de Petróleo y Gas (rho): {rho:.4f}')


--- PARÁMETROS CALIBRADOS (PROCESO OU LOGARÍTMICO) ---
Petróleo (WTI) -> Actual: $104.12 | Equilibrio LP: $75.26 | theta: 1.1372 | sigma: 45.04%
Gas Natural    -> Actual: $2.90 | Equilibrio LP: $4.05 | theta: 0.8449 | sigma: 58.26%
Correlación entre choques de Petróleo y Gas (rho): 0.0114


In [10]:
# ==============================================================================
# 4. SIMULACIÓN DE MONTE CARLO (CAMINOS ALEATORIOS CORRELACIONADOS)
# ==============================================================================
steps_per_year = int(round(1 / DT_SIM))
total_steps = N_YEARS * steps_per_year

np.random.seed(42)

# Matrices de log-precios X_t = ln(P_t): filas = pasos de tiempo, columnas = simulaciones
X_oil = np.zeros((total_steps + 1, N_SIMS))
X_gas = np.zeros((total_steps + 1, N_SIMS))

X_oil[0] = np.log(oil_params['last_price'])
X_gas[0] = np.log(gas_params['last_price'])

# Constantes de la solución exacta discretizada para el paso DT_SIM elegido
exp_oil = np.exp(-oil_params['theta'] * DT_SIM)
std_oil = oil_params['sigma'] * np.sqrt(
    (1 - np.exp(-2 * oil_params['theta'] * DT_SIM)) / (2 * oil_params['theta'])
)

exp_gas = np.exp(-gas_params['theta'] * DT_SIM)
std_gas = gas_params['sigma'] * np.sqrt(
    (1 - np.exp(-2 * gas_params['theta'] * DT_SIM)) / (2 * gas_params['theta'])
)

for t in range(1, total_steps + 1):
    # Ruido normal estándar independiente N(0,1)
    Z1 = np.random.standard_normal(N_SIMS)
    Z2 = np.random.standard_normal(N_SIMS)

    # Descomposición de Cholesky para correlacionar el ruido de Gas y Petróleo
    Z_oil = Z1
    Z_gas = rho * Z1 + np.sqrt(1 - rho**2) * Z2

    # Evolución exacta del proceso OU en log-precios
    X_oil[t] = (
        X_oil[t - 1] * exp_oil + oil_params['mu'] * (1 - exp_oil) + std_oil * Z_oil
    )
    X_gas[t] = (
        X_gas[t - 1] * exp_gas + gas_params['mu'] * (1 - exp_gas) + std_gas * Z_gas
    )

# Convertir de vuelta a precios en dólares P_t = exp(X_t)
P_oil_paths = np.exp(X_oil)
P_gas_paths = np.exp(X_gas)

# Si DT_SIM < 1 (ej. mensual), promediamos cada año para obtener precios anuales del DCF
# Si DT_SIM == 1, toma directamente los precios de cada año 1..N_YEARS
P_oil_annual = P_oil_paths[1:].reshape(N_YEARS, steps_per_year, N_SIMS).mean(axis=1)
P_gas_annual = P_gas_paths[1:].reshape(N_YEARS, steps_per_year, N_SIMS).mean(axis=1)

In [32]:
# ==============================================================================
# 5. CÁLCULO DEL DCF (FLUJO DE CAJA DESCONTADO) Y DISTRIBUCIÓN DEL NPV
# ==============================================================================
# Flujos de caja anuales (Ingresos por venta de Oil + Gas, sin otros costos por ahora)
FCF_annual = (P_oil_annual * OIL_PROD_ANNUAL) + (P_gas_annual * GAS_PROD_ANNUAL)

# Vector de factores de descuento: 1 / (1 + r)^t para t = 1, 2, ..., N_YEARS
years_vec = np.arange(1, N_YEARS + 1).reshape(-1, 1)
discount_factors = 1 / ((1 + WACC) ** years_vec)

# Valor Presente Neto (NPV) para cada una de las N_SIMS trayectorias (en Millones de $)
NPV_distribution = (FCF_annual * discount_factors).sum(axis=0) / 1e6

# Estadísticas clave del NPV
mean_npv = np.mean(NPV_distribution)
median_npv = np.median(NPV_distribution)
p10_npv = np.percentile(NPV_distribution, 10)
p90_npv = np.percentile(NPV_distribution, 90)
std_npv = np.std(NPV_distribution)

print(f'\n--- RESULTADOS DEL DCF ({N_SIMS:,} SIMULACIONES A {N_YEARS} AÑOS, r={WACC:.0%}) ---')
print(f'NPV Esperado (Media):   ${mean_npv:,.2f} M')
print(f'NPV Mediano (P50):      ${median_npv:,.2f} M')
print(f'Escenario Bajo (P10):   ${p10_npv:,.2f} M')
print(f'Escenario Alto (P90):   ${p90_npv:,.2f} M')
print(f'Desviación Estándar:    ${std_npv:,.2f} M')


--- RESULTADOS DEL DCF (200,000 SIMULACIONES A 10 AÑOS, r=10%) ---
NPV Esperado (Media):   $602.39 M
NPV Mediano (P50):      $598.81 M
Escenario Bajo (P10):   $525.57 M
Escenario Alto (P90):   $683.84 M
Desviación Estándar:    $62.15 M


In [33]:
# ==============================================================================
# 6. VISUALIZACIÓN INTERACTIVA EN PLOTLY (3 GRÁFICAS SEPARADAS)
# ==============================================================================
time_grid = np.linspace(0, N_YEARS, total_steps + 1)
n_paths_plot = 80  # Cantidad de trayectorias individuales a dibujar

In [34]:
# ------------------------------------------------------------------------------
# GRÁFICA 1: Trayectorias de Monte Carlo - Petróleo WTI
# ------------------------------------------------------------------------------
fig_oil = go.Figure()

# Caminos individuales (en gris/azul translúcido para no saturar)
for i in range(n_paths_plot):
    fig_oil.add_trace(
        go.Scatter(
            x=time_grid,
            y=P_oil_paths[:, i],
            mode='lines',
            line=dict(color='rgba(43, 92, 143, 0.20)', width=1),
            showlegend=(i == 0),
            name='Caminos aleatorios (muestra)',
            hoverinfo='skip',
        )
    )

# Media de las N_SIMS simulaciones
fig_oil.add_trace(
    go.Scatter(
        x=time_grid,
        y=P_oil_paths.mean(axis=1),
        mode='lines',
        line=dict(color='black', width=3),
        name=f'Media ({N_SIMS:,} simulaciones)',
    )
)

# Nivel de equilibrio de largo plazo
fig_oil.add_hline(
    y=oil_params['eq_price'],
    line_dash='dash',
    line_color='red',
    line_width=2,
    annotation_text=f"Equilibrio LP: ${oil_params['eq_price']:.2f}/bbl",
    annotation_position='top right',
)

fig_oil.update_layout(
    title=f'Simulación Monte Carlo: Precio del Petróleo WTI ({N_YEARS} años)',
    xaxis_title='Años',
    yaxis_title='Precio ($/bbl)',
    template='plotly_dark',
    width=950,
    height=500,
    hovermode='x unified',
)
fig_oil.show()

In [35]:
# ------------------------------------------------------------------------------
# GRÁFICA 2: Trayectorias de Monte Carlo - Gas Natural Henry Hub
# ------------------------------------------------------------------------------
fig_gas = go.Figure()

for i in range(n_paths_plot):
    fig_gas.add_trace(
        go.Scatter(
            x=time_grid,
            y=P_gas_paths[:, i],
            mode='lines',
            line=dict(color='rgba(0, 230, 118, 0.22)', width=1),
            showlegend=(i == 0),
            name='Caminos aleatorios (muestra)',
            hoverinfo='skip',
        )
    )

fig_gas.add_trace(
    go.Scatter(
        x=time_grid,
        y=P_gas_paths.mean(axis=1),
        mode='lines',
        line=dict(color='#ffffff', width=3),
        name=f'Media ({N_SIMS:,} simulaciones)',
    )
)

fig_gas.add_hline(
    y=gas_params['eq_price'],
    line_dash='dash',
    line_color='#ff5252',
    line_width=2,
    annotation_text=f"Equilibrio LP: ${gas_params['eq_price']:.2f}/mmBTU",
    annotation_position='top right',
    annotation_font_color='#ff5252',
)

fig_gas.update_layout(
    title=f'Simulación Monte Carlo: Precio del Gas Natural Henry Hub ({N_YEARS} años)',
    xaxis_title='Años',
    yaxis_title='Precio ($/mmBTU)',
    template='plotly_dark',
    width=950,
    height=500,
    hovermode='x unified',
)
fig_gas.show()

In [36]:
# ------------------------------------------------------------------------------
# GRÁFICA 3: Distribución de Frecuencias del NPV (DCF)
# ------------------------------------------------------------------------------
fig_npv = go.Figure()

fig_npv.add_trace(
    go.Histogram(
        x=NPV_distribution,
        nbinsx=70,
        marker_color='#3b82f6',
        marker_line_color='#111111',
        marker_line_width=0.8,
        opacity=0.88,
        name='Frecuencia NPV',
    )
)

# Líneas verticales para Media, P10 y P90
fig_npv.add_vline(
    x=mean_npv,
    line_dash='solid',
    line_color='#ff5252',
    line_width=2.5,
    annotation_text=f'Media: ${mean_npv:,.1f}M',
    annotation_position='top right',
    annotation_font_color='#ff5252',
)

fig_npv.add_vline(
    x=p10_npv,
    line_dash='dash',
    line_color='#ffb74d',
    line_width=2,
    annotation_text=f'P10: ${p10_npv:,.1f}M',
    annotation_position='top left',
    annotation_font_color='#ffb74d',
)

fig_npv.add_vline(
    x=p90_npv,
    line_dash='dash',
    line_color='#69f0ae',
    line_width=2,
    annotation_text=f'P90: ${p90_npv:,.1f}M',
    annotation_position='top right',
    annotation_font_color='#69f0ae',
)

fig_npv.update_layout(
    title=f'Distribución del Valor Presente Neto (NPV) — {N_SIMS:,} Simulaciones a {N_YEARS} Años (r = {WACC:.0%})',
    xaxis_title='NPV (Millones de USD)',
    yaxis_title='Frecuencia (N° de escenarios)',
    template='plotly_dark',
    width=950,
    height=500,
    bargap=0.05,
    showlegend=False,
)
fig_npv.show()